In [1]:
import numpy as np
from scipy.stats import norm
import pandas as pd
from statsmodels.stats.power import TTestIndPower

In [3]:
# Using statsmodels for power analysis
alpha = 0.05
power = 0.8
d = 0.5  # effect size (Cohen's d)

obj = TTestIndPower()
# For system-level transparency effect (balanced 2-level factor)
n = obj.solve_power(effect_size=d, alpha=alpha, power=power, 
                    ratio=1, alternative='two-sided') 

print(f"Required sample size per group: {n:.1f}")
print(f"Total participants needed: {2*n:.1f}")

# Power analysis for different effect sizes
print("\nSample size requirements for different effect sizes:")
print("-" * 50)
effect_sizes = [0.15, 0.3, 0.5]
for d in effect_sizes:
    n = obj.solve_power(effect_size=d, alpha=alpha, power=power, 
                        ratio=1, alternative='two-sided')
    print(f"Effect size d = {d}: {n:.0f} per group ({2*n:.0f} total)")

def power_analysis_calculator(alpha=0.05, power=0.80, target_d=0.5, verbose=True):
    """
    Calculate sample sizes for AI intervention study with two analyses.
    Uses statsmodels TTestIndPower for calculations.
    """
    
    obj = TTestIndPower()
    
    # Base calculation for balanced design
    base_n_per_group = obj.solve_power(effect_size=target_d, alpha=alpha, power=power, ratio=1, alternative='two-sided')
     
    if verbose:
        print(f"Power Analysis Parameters:")
        print(f"Alpha = {alpha}, Power = {power}, Target d = {target_d}")
        print(f"Base n per group = {base_n_per_group:.1f}")
        print()
    
    # Analysis A: Attribute effects (unbalanced: 20n vs 5n)
    # Use solve_power with ratio parameter for unbalanced design
    # ratio = n2/n1 = 5n/20n = 0.25 (low/high condition)
    n_high_group_a = obj.solve_power(effect_size=target_d, alpha=alpha, power=power, ratio=0.25, alternative='two-sided')
    # This gives us n for the HIGH condition (20n total)
    # So n_per_cell = n_high_group / 20, and low group = n_high_group / 4
    n_per_cell_a = int(np.ceil(n_high_group_a / 20))  # Divide by 20 scenarios in high condition
    n_low_total_a = int(np.ceil(n_high_group_a / 4))   # Low condition is 1/4 the size
    total_a = int(n_high_group_a + n_low_total_a)       # Total participants for Analysis A
    
    # Analysis B: Transparency effects (balanced: 5n vs 5n) 
    # Use solve_power with effective sample size
    n_per_group_b = obj.solve_power(effect_size=target_d, alpha=alpha, power=power, ratio=1, alternative='two-sided')
    # Each participant does 5 scenarios, so n_per_cell = n_per_group / 5
    n_per_cell_b = int(np.ceil(n_per_group_b / 5))
    additional_b = n_per_cell_b * 5  # 5 groups (5 high transparency)
    
    # Use larger requirement
    n_recommended = max(n_per_cell_a, n_per_cell_b)
    total_participants = n_recommended * 30  # 30 total unique groups
    
    # Calculate detectable effect sizes with recommended n
    # For Analysis A: use ratio=0.25 for unbalanced design
    eff_n_high_a = 20 * n_recommended  # High condition effective n
    detectable_d_a = obj.solve_power(nobs1=eff_n_high_a, alpha=alpha, power=power, ratio=0.25, alternative='two-sided')
    
    # For Analysis B: balanced design with 5n per group
    eff_n_b = 5 * n_recommended
    detectable_d_b = obj.solve_power(nobs1=eff_n_b, alpha=alpha, power=power, ratio=1, alternative='two-sided')
    
    if verbose:
        print(f"Analysis A (Prompt Attribute Effects - unbalanced 20n vs 5n):")
        print(f"  Required: {n_per_cell_a} per cell")
        print(f"  High condition total: {int(20 * n_per_cell_a)}")
        print(f"  Low condition total: {int(5 * n_per_cell_a)}")
        print(f"  Total: {total_a} participants \n")
        
        print(f"Analysis B (Transparency Effects - balanced 5n vs 5n):")
        print(f"  Required: {n_per_cell_b} per cell")  
        print(f"  Additional: {additional_b} participants")
        print()
        print(f"RECOMMENDATION: {n_recommended} per cell ({total_participants} total)")
        print(f"  Attribute effects: can detect d >= {detectable_d_a:.2f}")
        print(f"  Transparency effects: can detect d >= {detectable_d_b:.2f}")
        print()
    
    return {
        'n_per_cell_recommended': n_recommended,
        'total_participants': total_participants,
        'n_analysis_a': n_per_cell_a,
        'n_analysis_b': n_per_cell_b,
        'detectable_d_attributes': detectable_d_a,
        'detectable_d_transparency': detectable_d_b,
        'base_n': base_n_per_group
    }

# Standard analysis using statsmodels
print("\n2. STATSMODELS ANALYSIS (5 scenarios):")
print("-" * 40)
results = power_analysis_calculator(alpha=0.05, power=0.80, target_d=0.5)

Required sample size per group: 63.8
Total participants needed: 127.5

Sample size requirements for different effect sizes:
--------------------------------------------------
Effect size d = 0.15: 699 per group (1397 total)
Effect size d = 0.3: 175 per group (351 total)
Effect size d = 0.5: 64 per group (128 total)

2. STATSMODELS ANALYSIS (5 scenarios):
----------------------------------------
Power Analysis Parameters:
Alpha = 0.05, Power = 0.8, Target d = 0.5
Base n per group = 63.8

Analysis A (Prompt Attribute Effects - unbalanced 20n vs 5n):
  Required: 8 per cell
  High condition total: 160
  Low condition total: 40
  Total: 198 participants 

Analysis B (Transparency Effects - balanced 5n vs 5n):
  Required: 13 per cell
  Additional: 65 participants

RECOMMENDATION: 13 per cell (390 total)
  Attribute effects: can detect d >= 0.39
  Transparency effects: can detect d >= 0.50

